In [ ]:
import csv
from anarcii import Anarcii
import torch
import pandas as pd
import numpy as np
import os, time
from pathlib import Path
from itertools import islice
print(torch.cuda.is_available())  

True


In [ ]:
fasta_path = Path(r"D:\Thesis EGFR Round 3\Dataset\EG-combined ANARCII.fasta")
long_csv   = fasta_path.with_stem(fasta_path.stem + "_imgt_long").with_suffix(".csv")
wide_csv   = fasta_path.with_stem(fasta_path.stem + "_imgt_wide").with_suffix(".csv")

CHUNK      = 5000
USE_GPU    = True
MODE       = "speed"

def fasta_iter(fp: Path):
    head = None
    seq = []
    with fp.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if head:
                    yield head, "".join(seq)
                head = line[1:]
                seq = []
            else:
                seq.append(line)
        if head:
            yield head, "".join(seq)

def batched(it, n):
    it = iter(it)
    while True:
        batch = list(islice(it, n))
        if not batch:
            break
        yield batch
print(f"Loading Anarcii (GPU={USE_GPU})...")
model = Anarcii(seq_type="antibody", mode=MODE, cpu=not USE_GPU)
print(f"Using device: {model.device}")
with long_csv.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow([
        "Name", "Chain", "Score", "Query_start", "Query_end",
        "pos", "insertion", "aa", "error"
    ])
total = sum(1 for _ in fasta_iter(fasta_path))
total_chunks = (total + CHUNK - 1) // CHUNK
print(f"Total sequences: {total:,} | chunks: {total_chunks}")
start = time.time()
chunk_i = 0
for batch in batched(fasta_iter(fasta_path), CHUNK):
    chunk_i += 1
    t0 = time.time()
    tmp_fa = fasta_path.parent / f"_tmp_{chunk_i}.fasta"
    with tmp_fa.open("w", encoding="utf-8") as f:
        for name, seq in batch:
            f.write(f">{name}\n{seq}\n")
    results = model.number(str(tmp_fa))
    with long_csv.open("a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        for name, info in results.items():
            chain = info.get("chain_type", "")
            score = info.get("score", "")
            qs    = info.get("query_start", "")
            qe    = info.get("query_end", "")
            err   = info.get("error")
            numbering = info.get("numbering")
            if numbering is None:
                w.writerow([name, chain, score, qs, qe,
                            "", "", "", err])
            else:
                for ((pos, ins), aa) in numbering:
                    ins = (ins or " ").strip()  
                    w.writerow([name, chain, score, qs, qe,
                                pos, ins, aa, err])
    try:
        os.remove(tmp_fa)
    except OSError:
        pass
    dt = time.time() - t0
    pct = chunk_i / total_chunks * 100
    eta = (total_chunks - chunk_i) * dt
    print(f"[chunk {chunk_i}/{total_chunks}] {len(batch)} seqs | "
          f"{dt:.1f}s | {pct:.1f}% | ETA ~ {eta/60:.1f} min")
print(f"Step 1 done: long CSV written to {long_csv}")
print(f"Elapsed: {(time.time() - start)/60:.1f} min")

print("\nStep 2: building wide IMGT table from long CSV...")
df = pd.read_csv(long_csv)
df["pos"] = pd.to_numeric(df["pos"], errors="coerce")
meta_cols = ["Name", "Chain", "Score", "Query_start", "Query_end"]
meta = (
    df[meta_cols]
    .drop_duplicates(subset=["Name"])
    .set_index("Name")
)
df_valid = df[df["pos"].notna()].copy()
df_valid["pos_label"] = (
    df_valid["pos"].astype(int).astype(str)
    + df_valid["insertion"].fillna("").astype(str).str.strip()
)
labels = sorted(
    df_valid["pos_label"].unique(),
    key=lambda s: (
        int("".join(ch for ch in s if ch.isdigit())),
        "".join(ch for ch in s if ch.isalpha())
    )
)
wide_aa = (
    df_valid
    .pivot_table(index="Name", columns="pos_label", values="aa",
                 aggfunc="first")
)
for lab in labels:
    if lab not in wide_aa.columns:
        wide_aa[lab] = np.nan
wide_aa = wide_aa[labels]
wide = meta.join(wide_aa, how="left")
for lab in labels:
    if lab in wide.columns:
        wide[lab] = wide[lab].fillna("-")
wide = wide.reset_index()
wide = wide.rename(columns={
    "Query_start": "Query start",
    "Query_end": "Query end"
})
final_cols = ["Name", "Chain", "Score", "Query start", "Query end"] + labels
wide = wide[final_cols]
wide.to_csv(wide_csv, index=False)
print(f"Step 2 done: wide IMGT CSV written to {wide_csv}")

Loading Anarcii (GPU=True)...
Using device CUDA with 12 CPUs
Using device: cuda
Total sequences: 206,496 | chunks: 42
[chunk 1/42] 5000 seqs | 45.3s | 2.4% | ETA ~ 31.0 min
[chunk 2/42] 5000 seqs | 41.8s | 4.8% | ETA ~ 27.9 min
[chunk 3/42] 5000 seqs | 40.3s | 7.1% | ETA ~ 26.2 min
[chunk 4/42] 5000 seqs | 41.0s | 9.5% | ETA ~ 26.0 min
[chunk 5/42] 5000 seqs | 40.7s | 11.9% | ETA ~ 25.1 min
[chunk 6/42] 5000 seqs | 40.8s | 14.3% | ETA ~ 24.5 min
[chunk 7/42] 5000 seqs | 41.1s | 16.7% | ETA ~ 24.0 min
[chunk 8/42] 5000 seqs | 41.0s | 19.0% | ETA ~ 23.2 min
[chunk 9/42] 5000 seqs | 40.9s | 21.4% | ETA ~ 22.5 min
[chunk 10/42] 5000 seqs | 40.8s | 23.8% | ETA ~ 21.8 min
[chunk 11/42] 5000 seqs | 40.5s | 26.2% | ETA ~ 20.9 min
[chunk 12/42] 5000 seqs | 40.4s | 28.6% | ETA ~ 20.2 min
[chunk 13/42] 5000 seqs | 40.2s | 31.0% | ETA ~ 19.4 min
[chunk 14/42] 5000 seqs | 40.2s | 33.3% | ETA ~ 18.8 min
[chunk 15/42] 5000 seqs | 40.3s | 35.7% | ETA ~ 18.1 min
[chunk 16/42] 5000 seqs | 40.7s | 38.1% 

C:\Users\ankit\AppData\Local\Temp\ipykernel_26608\2468102237.py:110: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(long_csv)


Step 2 done: wide IMGT CSV written to D:\Thesis EGFR Round 3\Dataset\EG-combined ANARCII_imgt_wide.csv


Fix column insertions

In [ ]:
long_csv = r"D:\Thesis EGFR Round 3\Dataset\ANARCII\EG-combined ANARCII_imgt_long.csv"
wide_csv = long_csv.replace("_long", "_wide_fixed")

usecols = [
    "Name", "Chain", "Score", "Query_start", "Query_end",
    "pos", "insertion", "aa"
]

dtype_map = {
    "Name": "string",
    "Chain": "string",
    "Score": "float32",
    "Query_start": "Int32",
    "Query_end": "Int32",
    "insertion": "string",
    "aa": "string"
}

df = pd.read_csv(
    long_csv,
    usecols=usecols,
    dtype=dtype_map,
    low_memory=False
)

df["pos"] = pd.to_numeric(df["pos"], errors="coerce")
df_valid = df[df["pos"].notna()].copy()
df_valid["pos"] = df_valid["pos"].astype("Int32")
df_valid["insertion"] = df_valid["insertion"].fillna("").str.strip()
df_valid["pos_label"] = df_valid["pos"].astype(str) + df_valid["insertion"]
df_valid = df_valid[df_valid["pos_label"].str.contains(r"\d", na=False)].copy()
def parse_label(label):
    label = "" if pd.isna(label) else str(label).strip()
    digits = "".join(ch for ch in label if ch.isdigit())
    letters = "".join(ch for ch in label if ch.isalpha())

    if digits == "":
        return None, letters

    return int(digits), letters

def imgt_sort(label):
    base, ins = parse_label(label)

    if base is None:
        return (float("inf"), float("inf"))
    # correct the insertion patterns
    if base == 32:
        if ins == "":
            order = 0
        else:
            idx = "ABCDEFGHI".find(ins)
            order = idx + 1 if idx != -1 else 99
        return (32, order)
    
    if base == 33:
        if ins == "":
            return (33, 50)
        idx = "IHGFEDCBA".find(ins)
        order = idx + 1 if idx != -1 else 99
        return (33, order)
    
    if base == 111:
        if ins == "":
            order = 0
        else:
            idx = "ABCDEFGHI".find(ins)
            order = idx + 1 if idx != -1 else 99
        return (111, order)
    
    if base == 112:
        if ins == "":
            return (112, 50)
        idx = "IHGFEDCBA".find(ins)
        order = idx + 1 if idx != -1 else 99
        return (112, order)
    
    if ins == "":
        return (base, 0)

    return (base, ord(ins) - ord("A") + 1)

labels = sorted(df_valid["pos_label"].unique(), key=imgt_sort)
-
wide = (
    df_valid
    .drop_duplicates(subset=["Name", "pos_label"])
    .pivot(index="Name", columns="pos_label", values="aa")
)

wide = wide.reindex(columns=labels, fill_value="-")
wide = wide.fillna("-")

meta = (
    df[["Name", "Chain", "Score", "Query_start", "Query_end"]]
    .drop_duplicates(subset=["Name"])
    .set_index("Name")
)

wide = meta.join(wide, how="left").reset_index()

wide = wide.rename(columns={
    "Query_start": "Query start",
    "Query_end": "Query end"
})

wide.to_csv(wide_csv, index=False)
print("Written fixed IMGT wide file to:", wide_csv)

Written fixed IMGT wide file to: D:\Thesis EGFR Round 3\Dataset\ANARCII\EG-combined ANARCII_imgt_wide_fixed.csv


In [ ]:
model = Anarcii(seq_type="antibody", mode="speed", cpu=False)

test = """>test
QVQLQESGGGLVQAGGSLRLSCAASGSISGDGDMGWYRQAPGKERELVASIARGGSTNYADSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAAHAVLGYDHDYWGQGTQVTVSS
"""

with open("test.fa","w") as f:
    f.write(test)

result = model.number("test.fa")
print(result)


Using device CUDA with 12 CPUs
{'test': {'numbering': [((1, ' '), 'Q'), ((2, ' '), 'V'), ((3, ' '), 'Q'), ((4, ' '), 'L'), ((5, ' '), 'Q'), ((6, ' '), 'E'), ((7, ' '), 'S'), ((8, ' '), 'G'), ((9, ' '), 'G'), ((10, ' '), '-'), ((11, ' '), 'G'), ((12, ' '), 'L'), ((13, ' '), 'V'), ((14, ' '), 'Q'), ((15, ' '), 'A'), ((16, ' '), 'G'), ((17, ' '), 'G'), ((18, ' '), 'S'), ((19, ' '), 'L'), ((20, ' '), 'R'), ((21, ' '), 'L'), ((22, ' '), 'S'), ((23, ' '), 'C'), ((24, ' '), 'A'), ((25, ' '), 'A'), ((26, ' '), 'S'), ((27, ' '), 'G'), ((28, ' '), 'S'), ((29, ' '), 'I'), ((30, ' '), 'S'), ((31, ' '), '-'), ((32, ' '), '-'), ((33, ' '), '-'), ((34, ' '), '-'), ((35, ' '), 'G'), ((36, ' '), 'D'), ((37, ' '), 'G'), ((38, ' '), 'D'), ((39, ' '), 'M'), ((40, ' '), 'G'), ((41, ' '), 'W'), ((42, ' '), 'Y'), ((43, ' '), 'R'), ((44, ' '), 'Q'), ((45, ' '), 'A'), ((46, ' '), 'P'), ((47, ' '), 'G'), ((48, ' '), 'K'), ((49, ' '), 'E'), ((50, ' '), 'R'), ((51, ' '), 'E'), ((52, ' '), 'L'), ((53, ' '), 'V'), 